In [1]:
from __future__ import annotations

import json
from pathlib import Path

from agents.sandbox import FileMode, Group, Manifest, Permissions, SandboxPathGrant, User
from agents.sandbox.entries import Dir, File, LocalDir
from agents.sandbox.manifest import Environment

In [2]:
def resolve_paths(start: Path) -> tuple[Path, Path]:
    start = start.resolve()
    for path in (start, *start.parents):
        if path.name == "02-sandbox" and path.parent.name == "notebooks":
            return path.parents[1], path
        candidate = path / "notebooks" / "02-sandbox"
        if (candidate / "repo").is_dir():
            return path, candidate
    raise RuntimeError(f"Cannot resolve paths from {start}")


PROJECT_ROOT, EXAMPLE_DIR = resolve_paths(Path.cwd())
HOST_REPO_DIR = EXAMPLE_DIR / "repo"
HOST_SKILLS_DIR = EXAMPLE_DIR / "skills"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("EXAMPLE_DIR  =", EXAMPLE_DIR)
print("HOST_REPO_DIR=", HOST_REPO_DIR)
print("repo exists  =", HOST_REPO_DIR.is_dir())

PROJECT_ROOT = /Users/zhangtianzhu/Project/huya/ANIFORCE
EXAMPLE_DIR  = /Users/zhangtianzhu/Project/huya/ANIFORCE/notebooks/02-sandbox
HOST_REPO_DIR= /Users/zhangtianzhu/Project/huya/ANIFORCE/notebooks/02-sandbox/repo
repo exists  = True


In [3]:
def dump(obj):
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    return obj


def pretty(obj) -> None:
    print(json.dumps(obj, ensure_ascii=False, indent=2, default=str))


In [4]:
private_permissions = Permissions(
    owner=FileMode.READ | FileMode.WRITE,
    group=FileMode.NONE,
    other=FileMode.NONE,
)

output_permissions = Permissions(
    owner=FileMode.ALL,
    group=FileMode.ALL,
    other=FileMode.NONE,
)

manifest = Manifest(
    root="/workspace",
    users=[User(name="analyst")],
    environment=Environment(value={
        "DEMO_ENV": "manifest-env-visible-in-sandbox",
    }),
    entries={
        "repo": LocalDir(src=HOST_REPO_DIR),
        "workspace_notes/task.md": File(
            content=b"Read repo/task.md and write output/report.md.\n",
            permissions=private_permissions,
        ),
        "output": Dir(permissions=output_permissions),
    },
)

pretty({
    "root": manifest.root,
    "entries": {str(k): type(v).__name__ for k, v in manifest.entries.items()},
    "users": [u.name for u in manifest.users],
    "environment": dump(manifest.environment),
})

{
  "root": "/workspace",
  "entries": {
    "repo": "LocalDir",
    "workspace_notes/task.md": "File",
    "output": "Dir"
  },
  "users": [
    "analyst"
  ],
  "environment": {
    "value": {
      "DEMO_ENV": "manifest-env-visible-in-sandbox"
    }
  }
}


In [5]:
print(manifest.describe(depth=None))

/workspace
├── output/           # /workspace/output
├── repo/             # /workspace/repo
└── workspace_notes/  # /workspace/workspace_notes
    └── task.md       # /workspace/workspace_notes/task.md



In [6]:
def assert_manifest_entry_paths(m: Manifest) -> None:
    bad_paths = []
    for raw_path in m.entries:
        path = Path(raw_path)
        if path.is_absolute() or ".." in path.parts:
            bad_paths.append(str(raw_path))
    if bad_paths:
        raise ValueError(f"bad entry paths: {bad_paths}")


for label, entries in {
    "bad absolute": {"/tmp/report.md": File(content=b"bad")},
    "bad escape": {"../outside": Dir()},
    "good relative": {"output/report.md": File(content=b"ok")},
}.items():
    try:
        assert_manifest_entry_paths(Manifest(entries=entries))
        print(label, "=> OK")
    except ValueError as exc:
        print(label, "=>", exc)

bad absolute => bad entry paths: ['/tmp/report.md']
bad escape => bad entry paths: ['../outside']
good relative => OK


In [7]:
examples = {
    "private_file": Permissions(
        owner=FileMode.READ | FileMode.WRITE,
        group=FileMode.NONE,
        other=FileMode.NONE,
    ),
    "shared_output_dir": Permissions(
        owner=FileMode.ALL,
        group=FileMode.ALL,
        other=FileMode.NONE,
        directory=True,
    ),
    "public_readonly_file": Permissions(
        owner=FileMode.READ | FileMode.WRITE,
        group=FileMode.READ,
        other=FileMode.READ,
    ),
}

for name, permissions in examples.items():
    print(name, "=>", permissions, "mode=", oct(permissions.to_mode()))

private_file => -rw------- mode= 0o600
shared_output_dir => drwxrwx--- mode= 0o40770
public_readonly_file => -rw-r--r-- mode= 0o644


In [8]:
analyst = User(name="analyst")
reviewer = User(name="reviewer")
reviewers = Group(name="reviewers", users=[analyst, reviewer])

team_manifest = Manifest(
    root="/workspace",
    users=[analyst, reviewer],
    groups=[reviewers],
    entries={
        "shared": Dir(
            group=reviewers,
            permissions=Permissions(
                owner=FileMode.ALL,
                group=FileMode.READ | FileMode.EXEC,
                other=FileMode.NONE,
                directory=True,
            ),
        ),
        "private/task.md": File(
            content=b"only owner should read this\n",
            permissions=Permissions(
                owner=FileMode.READ | FileMode.WRITE,
                group=FileMode.NONE,
                other=FileMode.NONE,
            ),
        ),
    },
)

pretty({
    "users": [u.name for u in team_manifest.users],
    "groups": [{"name": g.name, "users": [u.name for u in g.users]} for g in team_manifest.groups],
    "entries": {str(k): type(v).__name__ for k, v in team_manifest.entries.items()},
})

{
  "users": [
    "analyst",
    "reviewer"
  ],
  "groups": [
    {
      "name": "reviewers",
      "users": [
        "analyst",
        "reviewer"
      ]
    }
  ],
  "entries": {
    "shared": "Dir",
    "private/task.md": "File"
  }
}


In [9]:
grant_manifest = Manifest(
    root="/workspace",
    entries={
        "repo": LocalDir(src=HOST_REPO_DIR),
        "output": Dir(),
    },
    extra_path_grants=(
        SandboxPathGrant(
            path="/opt/toolchain",
            read_only=True,
            description="trusted read-only runtime outside workspace",
        ),
    ),
)

pretty({
    "entries": {str(k): type(v).__name__ for k, v in grant_manifest.entries.items()},
    "extra_path_grants": dump(grant_manifest.extra_path_grants),
})

{
  "entries": {
    "repo": "LocalDir",
    "output": "Dir"
  },
  "extra_path_grants": [
    "path='/opt/toolchain' read_only=True description='trusted read-only runtime outside workspace'"
  ]
}


In [ ]:
summary = {
    "Manifest": "描述沙盒工作区启动配方",
    "File": "在沙盒里合成一个小文件",
    "Dir": "在沙盒里创建目录",
    "LocalDir": "把宿主机目录物化到沙盒里",
    "User": "声明沙盒用户",
    "Group": "声明沙盒用户组",
    "Permissions": "控制 owner/group/other 的读写执行权限",
    "SandboxPathGrant": "特批访问工作区外的可信绝对路径",
}
pretty(summary)

{
  "Manifest": "描述沙盒工作区启动配方",
  "File": "在沙盒里合成一个小文件",
  "Dir": "在沙盒里创建目录",
  "LocalDir": "把宿主机目录物化到沙盒里",
  "User": "声明沙盒用户",
  "Group": "声明沙盒用户组",
  "Permissions": "控制 owner/group/other 的读写执行权限",
  "SandboxPathGrant": "特批访问工作区外的可信绝对路径"
}


In [12]:
import io
import tarfile

from agents.sandbox import LocalSnapshotSpec, RemoteSnapshotSpec, resolve_snapshot

SNAPSHOT_BASE_DIR = PROJECT_ROOT / "drafts" / "260701" / "sandbox_snapshots"

local_snapshot_spec = LocalSnapshotSpec(base_path=SNAPSHOT_BASE_DIR)
remote_snapshot_spec = RemoteSnapshotSpec(client_dependency_key="my_snapshot_client")

pretty({
    "local_snapshot_spec": dump(local_snapshot_spec),
    "remote_snapshot_spec": dump(remote_snapshot_spec),
    "snapshot_base_dir": str(SNAPSHOT_BASE_DIR),
})

{
  "local_snapshot_spec": {
    "type": "local",
    "base_path": "/Users/zhangtianzhu/Project/huya/ANIFORCE/drafts/260701/sandbox_snapshots"
  },
  "remote_snapshot_spec": {
    "type": "remote",
    "client_dependency_key": "my_snapshot_client"
  },
  "snapshot_base_dir": "/Users/zhangtianzhu/Project/huya/ANIFORCE/drafts/260701/sandbox_snapshots"
}


In [15]:
snapshot_id = "demo-session-01"
local_snapshot = local_snapshot_spec.build(snapshot_id)
resolved_snapshot = resolve_snapshot(local_snapshot_spec, snapshot_id)

pretty({
    "local_snapshot": dump(local_snapshot),
    "resolved_snapshot": dump(resolved_snapshot),
    "expected_file": str(SNAPSHOT_BASE_DIR / f"{snapshot_id}.tar"),
})


{
  "local_snapshot": {
    "type": "local",
    "id": "demo-session-01",
    "base_path": "/Users/zhangtianzhu/Project/huya/ANIFORCE/drafts/260701/sandbox_snapshots"
  },
  "resolved_snapshot": {
    "type": "local",
    "id": "demo-session-01",
    "base_path": "/Users/zhangtianzhu/Project/huya/ANIFORCE/drafts/260701/sandbox_snapshots"
  },
  "expected_file": "/Users/zhangtianzhu/Project/huya/ANIFORCE/drafts/260701/sandbox_snapshots/demo-session-01.tar"
}


In [16]:
async def make_demo_workspace_tar() -> io.BytesIO:
    data = io.BytesIO()
    with tarfile.open(fileobj=data, mode="w") as tar:
        content = b"hello from snapshot\n"
        info = tarfile.TarInfo("output/report.md")
        info.size = len(content)
        tar.addfile(info, io.BytesIO(content))
    data.seek(0)
    return data

data = await make_demo_workspace_tar()
await local_snapshot.persist(data)
print("persisted=", SNAPSHOT_BASE_DIR / f"{snapshot_id}.tar")
print("restorable=", await local_snapshot.restorable())

restored = await local_snapshot.restore()
with tarfile.open(fileobj=restored, mode="r") as tar:
    print("tar names=", tar.getnames())
    report = tar.extractfile("output/report.md").read().decode()
    print("output/report.md=", report.strip())

persisted= /Users/zhangtianzhu/Project/huya/ANIFORCE/drafts/260701/sandbox_snapshots/demo-session-01.tar
restorable= True
tar names= ['output/report.md']
output/report.md= hello from snapshot


In [23]:
from agents.run import RunConfig
from agents.sandbox import SandboxRunConfig
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

# UnixLocalSandboxClient 不支持 manifest.users / manifest.groups，
# 因为那会变成在宿主机上创建系统用户/组。
local_lifecycle_manifest = Manifest(
    root="/workspace",
    environment=manifest.environment,
    entries=manifest.entries,
)

sdk_owned_run_config = RunConfig(
    sandbox=SandboxRunConfig(
        client=UnixLocalSandboxClient(),
        manifest=local_lifecycle_manifest,
        snapshot=local_snapshot_spec,
    ),
    tracing_disabled=True,
    workflow_name="sdk-owned sandbox lifecycle demo",
)

pretty({
    "mode": "SDK-owned",
    "sandbox_has_client": sdk_owned_run_config.sandbox.client is not None,
    "sandbox_has_session": sdk_owned_run_config.sandbox.session is not None,
    "manifest_entries": [str(k) for k in local_lifecycle_manifest.entries],
    "snapshot": dump(local_snapshot_spec),
})





{
  "mode": "SDK-owned",
  "sandbox_has_client": true,
  "sandbox_has_session": false,
  "manifest_entries": [
    "repo",
    "workspace_notes/task.md",
    "output"
  ],
  "snapshot": {
    "type": "local",
    "base_path": "/Users/zhangtianzhu/Project/huya/ANIFORCE/drafts/260701/sandbox_snapshots"
  }
}


In [24]:
client = UnixLocalSandboxClient()

# 这段会创建并启动一个真实本地 sandbox，但不调用模型。
# async with 退出时会自动走清理路径。
async with await client.create(
    manifest=local_lifecycle_manifest,
    snapshot=local_snapshot_spec,
) as sandbox:
    developer_owned_run_config = RunConfig(
        sandbox=SandboxRunConfig(session=sandbox),
        tracing_disabled=True,
        workflow_name="developer-owned sandbox lifecycle demo",
    )
    pretty({
        "mode": "developer-owned",
        "sandbox_type": type(sandbox).__name__,
        "sandbox_has_client": developer_owned_run_config.sandbox.client is not None,
        "sandbox_has_session": developer_owned_run_config.sandbox.session is not None,
        "state_type": type(sandbox.state).__name__,
    })

{
  "mode": "developer-owned",
  "sandbox_type": "SandboxSession",
  "sandbox_has_client": false,
  "sandbox_has_session": true,
  "state_type": "UnixLocalSandboxSessionState"
}


In [26]:
manual_sandbox = await client.create(
    manifest=local_lifecycle_manifest,
    snapshot=local_snapshot_spec,
)

try:
    await manual_sandbox.start()
    print("started manual sandbox")

    # 中途显式保存工作区检查点。
    await manual_sandbox.stop()
    print("stopped and persisted workspace snapshot")

    # 如果只是想显式持久化，也可以用：
    # await manual_sandbox.persist_workspace()
finally:
    await manual_sandbox.aclose()
    print("closed manual sandbox")

started manual sandbox
stopped and persisted workspace snapshot
closed manual sandbox
